# Encoding in Machine Learning

---

## What is Encoding?

Machine Learning models **cannot understand text or categories** — they only understand **numbers**.

So we convert categorical data (text) → numbers. This process is called **Encoding**.

**Example columns that need encoding:**

| Column | Values |
|---|---|
| City | Hyderabad, Chennai, Bengaluru |
| Education | BTech, MTech, PhD |

---
## Types of Categorical Data

Before encoding, we must identify **what kind** of categorical data we have.

### Type 1 — Ordinal Data
- Categories have a **meaningful order / relationship**
- Example: `BTech < MTech < PhD`
- ✅ Use → **Label Encoding**

### Type 2 — Nominal Data
- Categories have **NO order or relationship**
- Example: `Hyderabad, Chennai, Bengaluru` (no city is "greater" than another)
- ✅ Use → **One Hot Encoding**

---

### Comparison Table

| Feature | Ordinal | Nominal |
|---|---|---|
| Order exists? | ✅ Yes | ❌ No |
| Relationship? | ✅ Yes | ❌ No |
| Encoding method | Label Encoding | One Hot Encoding |
| Example | BTech, MTech, PhD | Hyderabad, Chennai, Bengaluru |

---
## Label Encoding — For Ordinal Data

Assign numbers to categories **while preserving the order**.

> **Rule:** Higher category → Higher number

**Example:**

| Education (Before) | Education_Encoded (After) |
|---|---|
| BTech | 0 |
| MTech | 1 |
| PhD | 2 |

✅ Order is preserved: `0 < 1 < 2` (just like `BTech < MTech < PhD`)

In [1]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Sample data
df_edu = pd.DataFrame({
    'Education': ['BTech', 'MTech', 'PhD', 'BTech', 'MTech']
})

print("--- Before Encoding ---")
print(df_edu)

# Step 1: Initialize LabelEncoder
le = LabelEncoder()

# Step 2: Fit and Transform
df_edu['Education_Encoded'] = le.fit_transform(df_edu['Education'])

print("\n--- After Label Encoding ---")
print(df_edu)

print("\n--- Mapping (Category → Number) ---")
for num, label in enumerate(le.classes_):
    print(f"  {label} → {num}")

--- Before Encoding ---
  Education
0     BTech
1     MTech
2       PhD
3     BTech
4     MTech

--- After Label Encoding ---
  Education  Education_Encoded
0     BTech                  0
1     MTech                  1
2       PhD                  2
3     BTech                  0
4     MTech                  1

--- Mapping (Category → Number) ---
  BTech → 0
  MTech → 1
  PhD → 2


---
## One Hot Encoding — For Nominal Data

Each category becomes its **own new column**.
- `1` = category is present ✅
- `0` = category is absent ❌

> **Rule:** 3 unique categories → 3 new columns

**Before:**

| Row | City |
|---|---|
| 1 | Hyderabad |
| 2 | Bengaluru |
| 3 | Chennai |
| 4 | Hyderabad |

**After One Hot Encoding:**

| Row | City_Hyderabad | City_Bengaluru | City_Chennai |
|---|---|---|---|
| 1 | 1 | 0 | 0 |
| 2 | 0 | 1 | 0 |
| 3 | 0 | 0 | 1 |
| 4 | 1 | 0 | 0 |

In [2]:
import pandas as pd

# Sample data
df_city = pd.DataFrame({
    'City': ['Hyderabad', 'Bengaluru', 'Chennai', 'Hyderabad']
})

print("--- Before Encoding ---")
print(df_city)

# One Hot Encoding using pandas get_dummies
city_encoded = pd.get_dummies(df_city['City'], prefix='City')

print("\n--- After One Hot Encoding ---")
print(city_encoded)

--- Before Encoding ---
        City
0  Hyderabad
1  Bengaluru
2    Chennai
3  Hyderabad

--- After One Hot Encoding ---
   City_Bengaluru  City_Chennai  City_Hyderabad
0           False         False            True
1            True         False           False
2           False          True           False
3           False         False            True


---
## Dummy Variable Trap

One Hot Encoding creates **one column per category**. But these columns are **perfectly dependent** on each other:

> If `City_Bengaluru = 0` and `City_Chennai = 0`, then `City_Hyderabad` **must** be `1`.

Mathematically, for every row:

`City_Hyderabad + City_Bengaluru + City_Chennai = 1`

This is **perfect multicollinearity**. In Linear Regression (which also has an intercept term, a column of all 1s), the model cannot tell the effect of each dummy apart from the intercept → coefficients become unstable / not uniquely solvable.

### ❌ The Trap
Keeping **all `n`** dummy columns for `n` categories.

### ✅ The Fix
Keep only **`n − 1`** columns. The dropped category becomes the **reference (baseline)**.

| City | City_Chennai | City_Hyderabad | Meaning |
|---|---|---|---|
| Bengaluru | 0 | 0 | baseline (both are 0) |
| Chennai | 1 | 0 | |
| Hyderabad | 0 | 1 | |

> **Rule:** `n` categories → `n − 1` dummy columns
>
> In pandas: `pd.get_dummies(..., drop_first=True)`  
> In sklearn: `OneHotEncoder(drop='first')`

**Note:** Tree-based models (Decision Tree, Random Forest) are not affected by this, but it's a good habit for linear models.

In [3]:
import pandas as pd
import numpy as np

df_trap = pd.DataFrame({'City': ['Hyderabad', 'Bengaluru', 'Chennai', 'Hyderabad']})

# ❌ Trap: keep ALL dummy columns
all_dummies = pd.get_dummies(df_trap['City'], prefix='City', dtype=int)
print("--- All dummy columns (trap) ---")
print(all_dummies)

# Proof of perfect multicollinearity: the columns always sum to 1
print("\nRow sums:", all_dummies.sum(axis=1).tolist())
print("Rank of matrix:", np.linalg.matrix_rank(all_dummies.values), "| Columns:", all_dummies.shape[1])
print("→ Rank < number of columns, so the columns are linearly dependent!")

# ✅ Fix: drop one column using drop_first=True
safe_dummies = pd.get_dummies(df_trap['City'], prefix='City', drop_first=True, dtype=int)
print("\n--- After drop_first=True (trap avoided) ---")
print(safe_dummies)
print("Rank of matrix:", np.linalg.matrix_rank(safe_dummies.values), "| Columns:", safe_dummies.shape[1])

# With sklearn:
# from sklearn.preprocessing import OneHotEncoder
# OneHotEncoder(drop='first', sparse_output=False).fit_transform(df_trap[['City']])

--- All dummy columns (trap) ---
   City_Bengaluru  City_Chennai  City_Hyderabad
0               0             0               1
1               1             0               0
2               0             1               0
3               0             0               1

Row sums: [1, 1, 1, 1]
Rank of matrix: 3 | Columns: 3
→ Rank < number of columns, so the columns are linearly dependent!

--- After drop_first=True (trap avoided) ---
   City_Chennai  City_Hyderabad
0             0               1
1             0               0
2             1               0
3             0               1
Rank of matrix: 2 | Columns: 2


---
## Full Example — Both Encodings Together

Let's apply **both** encodings on a dataset that has:
- `City` → Nominal → One Hot Encoding
- `Education` → Ordinal → Label Encoding

In [4]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Original Dataset
df = pd.DataFrame({
    'City':      ['Hyderabad', 'Chennai',  'Bengaluru', 'Hyderabad'],
    'Education': ['BTech',     'MTech',    'PhD',       'MTech']
})

print("--- Original Dataset ---")
print(df)

# Step 1: Label Encoding for Ordinal column (Education)
le = LabelEncoder()
df['Education_Encoded'] = le.fit_transform(df['Education'])
print("\n--- After Label Encoding (Education) ---")
print(df)

# Step 2: One Hot Encoding for Nominal column (City)
city_encoded = pd.get_dummies(df['City'], prefix='City')
print("\n--- After One Hot Encoding (City) ---")
print(city_encoded)
# Step 3: Merge encoded columns with original dataframe
df_final = pd.concat([df, city_encoded], axis=1)

# Step 4: Drop original categorical columns (no longer needed)
df_final = df_final.drop(columns=['City', 'Education'])

print("\n--- Final Dataset (Fully Numerical) ---")
print(df_final)

--- Original Dataset ---
        City Education
0  Hyderabad     BTech
1    Chennai     MTech
2  Bengaluru       PhD
3  Hyderabad     MTech

--- After Label Encoding (Education) ---
        City Education  Education_Encoded
0  Hyderabad     BTech                  0
1    Chennai     MTech                  1
2  Bengaluru       PhD                  2
3  Hyderabad     MTech                  1

--- After One Hot Encoding (City) ---
   City_Bengaluru  City_Chennai  City_Hyderabad
0           False         False            True
1           False          True           False
2            True         False           False
3           False         False            True

--- Final Dataset (Fully Numerical) ---
   Education_Encoded  City_Bengaluru  City_Chennai  City_Hyderabad
0                  0           False         False            True
1                  1           False          True           False
2                  2            True         False           False
3                  1

---
## Common Mistakes to Avoid

### ❌ Wrong: Using One Hot Encoding on Ordinal data
- **Problem:** You lose the order — the PhD > MTech > BTech relationship gets destroyed.

### ❌ Wrong: Using Label Encoding on Nominal data
- **Problem:** The model assumes `Bengaluru(2) > Chennai(1) > Hyderabad(0)` — which is completely wrong!

---

## Golden Rule

> **Ask yourself: "Do the categories have an order / relationship?"**
>
> - **YES** → Label Encoding
> - **NO** → One Hot Encoding

### See the mistake in action — why Label Encoding is wrong for Nominal data

In [5]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df_wrong = pd.DataFrame({'City': ['Hyderabad', 'Chennai', 'Bengaluru', 'Hyderabad']})

le = LabelEncoder()
df_wrong['City_LabelEncoded'] = le.fit_transform(df_wrong['City'])

print("❌ Label Encoding on Nominal (City) — WRONG approach:")
print(df_wrong)
print("\nThe model will wrongly think Hyderabad(2) > Chennai(1) > Bengaluru(0)!")
print("Cities have no order — this is misleading to the model.")

❌ Label Encoding on Nominal (City) — WRONG approach:
        City  City_LabelEncoded
0  Hyderabad                  2
1    Chennai                  1
2  Bengaluru                  0
3  Hyderabad                  2

The model will wrongly think Hyderabad(2) > Chennai(1) > Bengaluru(0)!
Cities have no order — this is misleading to the model.


---
## Quick Summary — 5 Steps to Encode Any Dataset

1. **Identify** categorical columns in your dataset
2. **Check** if categories have order → Ordinal or Nominal?
3. **Ordinal?** → Use `LabelEncoder` from `sklearn`
4. **Nominal?** → Use `pd.get_dummies()` from `pandas`
5. **Merge** encoded columns & **drop** original columns → All numerical ✅

---

## Key Terms

| Term | Meaning |
|---|---|
| Encoding | Converting categories to numbers |
| Ordinal Data | Categorical data **with** order/relationship |
| Nominal Data | Categorical data **without** order |
| Label Encoding | Assigns numbers 0, 1, 2... preserving order |
| One Hot Encoding | Creates new columns with 1s and 0s |
| `pd.get_dummies()` | Pandas function for One Hot Encoding |
| `LabelEncoder` | Sklearn class for Label Encoding |
| `fit_transform()` | Fits the data and transforms it in one step |

---
## Practice — Try It Yourself!

Apply the correct encoding to this dataset:

| Name | Department | Experience |
|---|---|---|
| Alice | HR | Junior |
| Bob | IT | Senior |
| Carol | Finance | Mid |
| Dave | IT | Junior |

- `Department` → Nominal (HR, IT, Finance — no order) → One Hot Encoding
- `Experience` → Ordinal (Junior < Mid < Senior) → Label Encoding

In [13]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Dataset
df_practice = pd.DataFrame({
    'Name':       ['Alice', 'Bob',    'Carol',   'Dave'],
    'Department': ['HR',    'IT',     'Finance',  'IT'],
    'Experience': ['Junior','Senior', 'Mid',      'Junior']
})

print("--- Original Dataset ---")
print(df_practice)

# TODO: Apply Label Encoding to 'Experience'
# Hint: Junior=0, Mid=1, Senior=2
le = LabelEncoder()
df_practice['Experience_LabelEncoded'] = le.fit_transform(df_practice['Experience'])
df_practice
# # TODO: Apply One Hot Encoding to 'Department'
df = pd.get_dummies(df_practice['Department'], prefix='Department')
# TODO: Merge and drop original columns
df_final = pd.concat([df_practice, df], axis=1)
df_final = df_final.drop(columns=['Department', 'Experience'])
df_final


--- Original Dataset ---
    Name Department Experience
0  Alice         HR     Junior
1    Bob         IT     Senior
2  Carol    Finance        Mid
3   Dave         IT     Junior


,Name,Experience_LabelEncoded,Department_Finance,Department_HR,Department_IT
0,Alice,0,False,True,False
1,Bob,2,False,False,True
2,Carol,1,True,False,False
3,Dave,0,False,False,True
